In [691]:
%%capture
# see comments in README on changes to the conda venv
import os
from pathlib import Path

# import modin.pandas as pd
import pandas as pd
from dj_notebook import activate

env_file = os.environ["INTE_ENV"]
#analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
#reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [692]:
import numpy as np
import pandas as pd
from pathlib import Path

In [693]:
folder = Path("/Users/erikvw/Documents/ucl/protocols/inte-africa/export/raw_tables/csv")
remote_stata_folder = Path("/Users/erikvw/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Documents - igh.respond-africa/INTE-AFRICA-data/final database archive/exported/stata/20230316/")

In [694]:
df_dd = pd.read_csv(folder / "edc_data_manager_datadictionary.csv", index_col=0, delimiter="|")

In [695]:
df_inte_scr = pd.read_csv(folder / "inte_screening_subjectscreening.csv", index_col=0, delimiter="|")
df_inte_scr = df_inte_scr.rename(columns={"gender_x": "gender"})
df_inte_scr["hiv"] = 1
df_inte_scr["htn"] = np.where(df_inte_scr["clinic_type"] =="ncd_clinic", 1, 0)
df_inte_scr["dm"] = np.where(df_inte_scr["clinic_type"] =="diabetes_clinic", 1, 0)
df_inte_scr = df_inte_scr.drop(columns=["gender_y"])
df_inte_scr["multi"] = np.where(df_inte_scr.hiv+df_inte_scr.htn+df_inte_scr.dm>1 , 1, 0)
# df_inte_scr[["multi", "hiv", "htn", "dm", "clinic_type", "selection_method", "qualifying_condition", "age_in_years"]]
# df_inte_scr.multi.value_counts()


In [696]:
excluded = ["103-113-0164-1","103-113-0057-7","103-103-0221-0","103-109-0221-7","103-103-0140-2","103-208-0228-2","103-104-0008-9","103-112-0090-0", "103-103-0106-3","103-107-0212-0","103-104-0163-2","103-103-0069-3","103-107-0058-7","103-104-0076-6","103-103-0131-1","103-104-0210-1", "103-104-0161-6","103-110-0194-4"]
enrolled_7029 = pd.read_stata("/Users/erikvw/Documents/ucl/protocols/inte-africa/stata/subjects_7029.dta")
enrolled_7029 = enrolled_7029[["subject_identifier"]].copy().reset_index(drop=True)
len(enrolled_7029[enrolled_7029["subject_identifier"].isin(excluded)]) == 0
# enrolled_7029

True

In [697]:
df_consent = pd.read_stata(remote_stata_folder / "inte_consent_subjectconsent_20230316.dta").drop(columns=["index"]).reset_index(drop=True)
df_consent = df_consent[df_consent["subject_identifier"].isin(enrolled_7029["subject_identifier"])].copy().reset_index(drop=True)
df_consent["dob"] = pd.to_datetime(df_consent["dob"])
df_consent["consent_datetime"] = pd.to_datetime(df_consent["consent_datetime"])
df_consent["age_in_years"] = np.round((df_consent["consent_datetime"] - df_consent["dob"]).dt.days / 365.25).astype(int)
df_consent["country"] = df_consent.site_id.apply(lambda s: "uganda" if s <200 else "tanzania")
df_consent["study"] = "inte_africa"
df_rando = pd.read_csv("/Users/erikvw/Documents/ucl/protocols/inte-africa/export/randomizationlist.csv")
df_rando = df_rando[df_rando["subject_identifier"].isin(enrolled_7029["subject_identifier"])].copy().reset_index(drop=True)
df_consent = df_consent.merge(df_rando[["subject_identifier", "assignment"]], on="subject_identifier", how="left")
print(len(df_consent))

7029


In [698]:
df_inte = df_consent[["subject_identifier", "assignment", "gender", "dob", "age_in_years", 'site_id', "country", "study"]].copy().reset_index(drop=True)

In [699]:
df_visit = pd.read_stata(remote_stata_folder / "inte_subject_subjectvisit_20230316.dta").reset_index(drop=True)
df_visit = df_visit[df_visit["subject_identifier"].isin(enrolled_7029["subject_identifier"])].copy().reset_index(drop=True)
df_visit = df_visit.drop(columns=["index"]).rename(columns={"visit_code": "visit_code_str", "id": "subject_visit_id", "report_datetime": "visit_datetime", "reason": "visit_reason"})
df_visit["visit_datetime"] = pd.to_datetime(df_visit["visit_datetime"])
df_visit["site_id"] =df_visit["site_id"].astype("int64")
df_visit["visit_code"] = pd.to_numeric(df_visit["visit_code_str"]) + df_visit["visit_code_sequence"]/10
df_visit = df_visit[["subject_visit_id", "subject_identifier", "visit_code", "visit_datetime", "visit_reason", "reason_unscheduled", "reason_unscheduled_other", "site_id"]].copy()
# df_visit["visit_code"] =df_visit["visit_code"].astype("Float64")
df_visit["reason_unscheduled"] = np.where(df_visit["reason_unscheduled"] =='OTHER', df_visit["reason_unscheduled_other"], df_visit["reason_unscheduled"])
df_visit["reason_unscheduled"] = np.where(pd.isna(df_visit["reason_unscheduled"]), "", df_visit["reason_unscheduled"])
df_visit["reason_unscheduled_other"] = np.where(pd.isna(df_visit["reason_unscheduled_other"]), "", df_visit["reason_unscheduled_other"])

df_visit = df_visit.merge(df_visit.sort_values(by=["subject_identifier", "visit_datetime"], ascending=True).groupby("subject_identifier").agg("first").reset_index().rename(columns={"visit_datetime": "baseline_datetime"})[["subject_identifier", "baseline_datetime"]], on="subject_identifier", how="left")
df_visit = df_visit.merge(df_visit.sort_values(by=["subject_identifier", "visit_datetime"], ascending=True).groupby("subject_identifier").agg("last").reset_index().rename(columns={"visit_datetime": "endline_datetime"})[["subject_identifier", "endline_datetime"]], on="subject_identifier", how="left")
df_visit["days_on_study"] = (df_visit["endline_datetime"] - df_visit["baseline_datetime"]).dt.days

df_visit = df_visit.sort_values(by=["subject_identifier", "visit_datetime"], ascending=True).reset_index(drop=True)


In [700]:
# get initial review - HIV
# flag post baseline Diagnoses
df_hiv = pd.read_stata(remote_stata_folder / "inte_subject_hivinitialreview_20230316.dta").drop(columns=["index"]).reset_index(drop=True)
df_hiv = df_hiv[df_hiv["subject_identifier"].isin(enrolled_7029["subject_identifier"])].copy().reset_index(drop=True)

df_hiv["hiv"] = 1
df_hiv["dx_date"] = pd.to_datetime(df_hiv["dx_date"])
df_hiv["dx_estimated_date"] = pd.to_datetime(df_hiv["dx_estimated_date"])
df_hiv["hiv_dx_date"] = df_hiv["dx_date"].fillna(df_hiv["dx_estimated_date"])

df_vl_from_initialreview = df_hiv[["subject_identifier", "vl", "vl_date"]].rename(columns={"vl": "result", "vl_date": "drawn_date"}).copy().reset_index(drop=True)

In [701]:
df_vl = pd.read_stata(remote_stata_folder / 'inte_subject_viralloadresult_20230316.dta')[["subject_identifier", "drawn_date", "result"]].reset_index(drop=True)
df_vl = pd.concat([df_vl, df_vl_from_initialreview]).reset_index(drop=True)

In [702]:
df_vl = pd.read_stata(remote_stata_folder / 'inte_subject_viralloadresult_20230316.dta')[["subject_identifier", "drawn_date", "result"]].reset_index(drop=True)
df_vl = pd.concat([df_vl, df_vl_from_initialreview]).reset_index(drop=True)
df_vl = df_vl.merge(df_visit[["subject_identifier", "baseline_datetime", "endline_datetime"]], on="subject_identifier", how="left")
df_vl["drawn_date"]  = pd.to_datetime(df_vl["drawn_date"])
df_vl["baseline_datetime"]  = pd.to_datetime(df_vl["baseline_datetime"])
df_vl["vl_days"] = np.where(~df_vl["drawn_date"].isna(), (df_vl["drawn_date"] - df_vl["baseline_datetime"]).dt.days, np.nan)
df_vl_endline = (
    df_vl[df_vl["vl_days"]>=182]
    .sort_values(by=["subject_identifier","drawn_date"], ascending=[True, False])
    .drop_duplicates(subset=["subject_identifier"], keep="first")
    .rename(columns={"result":"vl_endline", "drawn_date":"vl_date_endline", "vl_days": "vl_days_endline"})
    .reset_index(drop=True)
    [["subject_identifier","vl_date_endline", "vl_endline", "vl_days_endline"]]
)
df_vl_baseline = (
    df_vl[df_vl["vl_days"]<182]
    .sort_values(by=["subject_identifier","drawn_date"], ascending=[True, True])
    .drop_duplicates(subset=["subject_identifier"], keep="first")
    .rename(columns={"result":"vl_baseline", "drawn_date":"vl_date_baseline", "vl_days": "vl_days_baseline"})
    .reset_index(drop=True)
    [["subject_identifier","vl_date_baseline", "vl_baseline", "vl_days_baseline"]]
)

In [703]:
df_inte = df_inte.merge(df_vl_endline, on=["subject_identifier"], how="left")
df_inte = df_inte.merge(df_vl_baseline, on=["subject_identifier"], how="left")
df_inte = df_inte.merge(df_hiv[["subject_identifier", "hiv_dx_date", "hiv"]], on=["subject_identifier"], how="left")
df_inte["hiv"] = np.where(df_inte["hiv"]==1, 1, 0)
print(len(df_inte))

7029


In [704]:
# df_inte[df_inte["hiv"]==1]

In [705]:
# get initial review - HTN
# flag post baseline Diagnoses
df_htn = pd.read_stata(remote_stata_folder / "inte_subject_htninitialreview_20230316.dta").drop(columns=["index"]).reset_index(drop=True)
df_htn = df_htn[df_htn["subject_identifier"].isin(enrolled_7029["subject_identifier"])].copy().reset_index(drop=True)
df_htn["htn"] = 1
df_htn["htn_dx_date"] = df_htn["dx_date"].fillna(df_htn["dx_estimated_date"])
df_htn = df_htn.sort_values(by=["subject_identifier","htn_dx_date"], ascending=[True, True]).drop_duplicates(subset=["subject_identifier"], keep="first").reset_index(drop=True)

In [706]:
df_bp = pd.read_stata(remote_stata_folder / "inte_subject_indicators_20230316.dta").drop(columns=["index"]).reset_index(drop=True)
df_bp = (
    df_bp[df_bp["subject_identifier"].isin(enrolled_7029["subject_identifier"])]
    .copy()
    .reset_index(drop=True)
    .query("subject_identifier.isin(@df_htn.subject_identifier)")
    [["subject_identifier", "subject_visit_id", "report_datetime", "sys_blood_pressure_r1", "dia_blood_pressure_r1", "sys_blood_pressure_r2", "dia_blood_pressure_r2"]]
)

df_bp["sys_blood_pressure_avg"] = np.where(df_bp["sys_blood_pressure_r2"].isna(), df_bp["sys_blood_pressure_r1"], (df_bp["sys_blood_pressure_r1"]+ df_bp["sys_blood_pressure_r2"])/2)
df_bp["dia_blood_pressure_avg"] = np.where(df_bp["dia_blood_pressure_r2"].isna(), df_bp["dia_blood_pressure_r1"], (df_bp["dia_blood_pressure_r1"]+ df_bp["dia_blood_pressure_r2"])/2)
df_bp = df_bp.drop(columns=["sys_blood_pressure_r1", "sys_blood_pressure_r2", "dia_blood_pressure_r1", "sys_blood_pressure_r2"])
df_bp = df_bp.merge(df_visit[["subject_identifier", "baseline_datetime", "endline_datetime"]], on="subject_identifier", how="left")
df_bp["bp_days"] = (df_bp["report_datetime"] - df_bp["baseline_datetime"]).dt.days

df_bp_baseline = (
    df_bp[df_bp["bp_days"]<182]
    .sort_values(by=["subject_identifier","report_datetime"], ascending=[True, True])
    .drop_duplicates(subset=["subject_identifier"], keep="first")
    .rename(columns={"sys_blood_pressure_avg":"bp_sys_baseline", "dia_blood_pressure_avg":"bp_dia_baseline", "report_datetime":"bp_date_baseline", "bp_days": "bp_days_baseline"})
    .reset_index(drop=True)
    [["subject_identifier","bp_date_baseline", "bp_sys_baseline", "bp_dia_baseline", "bp_days_baseline"]]
)
print(df_bp_baseline.subject_identifier.nunique())

df_bp_endline = (
    df_bp[df_bp["bp_days"]>=182]
    .sort_values(by=["subject_identifier","report_datetime"], ascending=[True, False])
    .drop_duplicates(subset=["subject_identifier"], keep="first")
    .rename(columns={"sys_blood_pressure_avg":"bp_sys_endline", "dia_blood_pressure_avg":"bp_dia_endline", "report_datetime":"bp_date_endline", "bp_days": "bp_days_endline"})
    .reset_index(drop=True)
    [["subject_identifier","bp_date_endline", "bp_sys_endline", "bp_dia_endline", "bp_days_endline"]]
)

3322


In [707]:
df_inte = df_inte.merge(df_bp_endline, on=["subject_identifier"], how="left")
df_inte = df_inte.merge(df_bp_baseline, on=["subject_identifier"], how="left")
df_inte = df_inte.merge(df_htn[["subject_identifier", "htn_dx_date", "htn"]], on=["subject_identifier"], how="left")
df_inte["htn"] = np.where(df_inte["htn"]==1, 1, 0)
print(len(df_inte))

7029


In [708]:
# get initial review - DM
df_dm = pd.read_stata(remote_stata_folder / "inte_subject_dminitialreview_20230316.dta").drop(columns=["index"]).reset_index(drop=True)
df_dm = df_dm[df_dm["subject_identifier"].isin(enrolled_7029["subject_identifier"])].copy().reset_index(drop=True)
df_dm["dm"] = 1
df_dm["dm_dx_date"] = df_dm["dx_date"].fillna(df_dm["dx_estimated_date"])
df_dm = df_dm.sort_values(by=["subject_identifier","dm_dx_date"], ascending=[True, True]).drop_duplicates(subset=["subject_identifier"], keep="first").reset_index(drop=True)

In [709]:
df_glu = (
    df_dm[(df_dm["glucose_fasted"]=="Yes") & ~(df_dm["glucose"].isna())]
    .copy()
    .reset_index()
    [["subject_identifier", "glucose_date", "glucose", "glucose_units"]]
)
df_glu2 = (
    pd.read_stata(remote_stata_folder / "inte_subject_glucose_20230316.dta")
    .drop(columns=["index"]).reset_index(drop=True)
    .query("glucose_fasted=='Yes'")
    [["subject_identifier", "glucose_date", "glucose", "glucose_units"]]
)
df_glu = pd.concat([df_glu, df_glu2]).reset_index(drop=True)
df_glu["glucose_date"] = pd.to_datetime(df_glu["glucose_date"])
df_glu = df_glu.merge(df_visit[["subject_identifier", "baseline_datetime", "endline_datetime"]], on="subject_identifier", how="left")
df_glu["glucose_days"] = (df_glu["glucose_date"] - df_glu["baseline_datetime"]).dt.days
df_glu_baseline = (
    df_glu[df_glu["glucose_days"]<182]
    .sort_values(by=["subject_identifier","glucose_date"], ascending=[True, True])
    .drop_duplicates(subset=["subject_identifier"], keep="first")
    .rename(columns={"glucose":"glucose_baseline", "glucose_date":"glucose_date_baseline", "glucose_days": "glucose_days_baseline"})
    .reset_index(drop=True)
    [["subject_identifier","glucose_date_baseline", "glucose_baseline","glucose_days_baseline"]]
)
df_glu_endline = (
    df_glu[df_glu["glucose_days"]>=182]
    .sort_values(by=["subject_identifier","glucose_date"], ascending=[True, False])
    .drop_duplicates(subset=["subject_identifier"], keep="first")
    .rename(columns={"glucose":"glucose_endline", "glucose_date":"glucose_date_endline", "glucose_days": "glucose_days_endline"})
    .reset_index(drop=True)
    [["subject_identifier","glucose_date_endline", "glucose_endline","glucose_days_endline"]]
)

In [710]:
df_inte = df_inte.merge(df_glu_endline, on=["subject_identifier"], how="left")
df_inte = df_inte.merge(df_glu_baseline, on=["subject_identifier"], how="left")
df_inte = df_inte.merge(df_dm[["subject_identifier", "dm_dx_date", "dm"]], on=["subject_identifier"], how="left")
df_inte["dm"] = np.where(df_inte["dm"]==1, 1, 0)
print(len(df_inte))

7029


In [711]:
df_inte["multi"] = np.where(df_inte["hiv"]+df_inte["htn"]+df_inte["dm"]>1, df_inte["hiv"]+df_inte["htn"]+df_inte["dm"], 0)
# df_inte[(df_inte["multi"]>1) & (df_inte["hiv"]==1)][["subject_identifier", "vl_endline", "bp_sys_endline", "bp_dia_endline", "glucose_endline", "hiv", "htn", "dm", "multi"]]

In [712]:
df_inte["glucose_baseline"] = pd.to_numeric(df_inte["glucose_baseline"], errors="coerce")
df_inte["bp_sys_baseline"] = pd.to_numeric(df_inte["bp_sys_baseline"], errors="coerce")
df_inte["bp_dia_baseline"] = pd.to_numeric(df_inte["bp_dia_baseline"], errors="coerce")
df_inte["vl_baseline"] = pd.to_numeric(df_inte["vl_baseline"], errors="coerce")

df_inte["bp_controlled_baseline"] = np.where(((df_inte["bp_sys_baseline"]<140) & (df_inte["bp_dia_baseline"]<90)), 1, 0)
df_inte["bp_controlled_baseline"] = np.where(df_inte["htn"]==1, df_inte["bp_controlled_baseline"], np.nan)

df_inte["glucose_controlled_baseline"] = np.where(df_inte["glucose_baseline"]<7.0, 1, 0)
df_inte["glucose_controlled_baseline"] = np.where(df_inte["dm"]==1, df_inte["glucose_controlled_baseline"], np.nan)

df_inte["vl_controlled_baseline"] = np.where(df_inte["vl_baseline"]<1000.0, 1, 0)
df_inte["vl_controlled_baseline"] = np.where(df_inte["hiv"]==1, df_inte["vl_controlled_baseline"], np.nan)

df_inte["glucose_endline"] = pd.to_numeric(df_inte["glucose_endline"], errors="coerce")
df_inte["bp_sys_endline"] = pd.to_numeric(df_inte["bp_sys_endline"], errors="coerce")
df_inte["bp_dia_endline"] = pd.to_numeric(df_inte["bp_dia_endline"], errors="coerce")
df_inte["vl_endline"] = pd.to_numeric(df_inte["vl_endline"], errors="coerce")

df_inte["bp_controlled_endline"] = np.where(((df_inte["bp_sys_endline"]<140) & (df_inte["bp_dia_endline"]<90)), 1, 0)
df_inte["bp_controlled_endline"] = np.where(df_inte["htn"]==1, df_inte["bp_controlled_endline"], np.nan)

df_inte["glucose_controlled_endline"] = np.where(df_inte["glucose_endline"]<7.0, 1, 0)
df_inte["glucose_controlled_endline"] = np.where(df_inte["dm"]==1, df_inte["glucose_controlled_endline"], np.nan)

df_inte["vl_controlled_endline"] = np.where(df_inte["vl_endline"]<1000.0, 1, 0)
df_inte["vl_controlled_endline"] = np.where(df_inte["hiv"]==1, df_inte["vl_controlled_endline"], np.nan)

In [713]:
df_inte["cohort"] = ""
df_inte["cohort"] = np.where((df_inte["hiv"]==1) & (df_inte["htn"]==1) & (df_inte["dm"]==1), "HIV_HTN_DM", df_inte["cohort"])
df_inte["cohort"] = np.where((df_inte["hiv"]==1) & (df_inte["htn"]==1) & (df_inte["dm"]==0), "HIV_HTN", df_inte["cohort"])
df_inte["cohort"] = np.where((df_inte["hiv"]==1) & (df_inte["htn"]==0) & (df_inte["dm"]==1), "HIV_HTN", df_inte["cohort"])
df_inte["cohort"] = np.where((df_inte["hiv"]==1) & (df_inte["htn"]==0) & (df_inte["dm"]==0), "HIV_ALONE", df_inte["cohort"])
df_inte["cohort"] = np.where((df_inte["hiv"]==0) & (df_inte["htn"]==1) & (df_inte["dm"]==0), "HTN_ALONE", df_inte["cohort"])
df_inte["cohort"] = np.where((df_inte["hiv"]==0) & (df_inte["htn"]==1) & (df_inte["dm"]==1), "HTN_DM", df_inte["cohort"])
df_inte["cohort"] = np.where((df_inte["hiv"]==0) & (df_inte["htn"]==0) & (df_inte["dm"]==1), "DM_ALONE", df_inte["cohort"])
df_inte["cohort"].value_counts(dropna=False)

cohort
HIV_ALONE     3187
HTN_ALONE     1754
HTN_DM         848
HIV_HTN        701
DM_ALONE       424
HIV_HTN_DM     115
Name: count, dtype: int64

In [714]:
df_inte

,subject_identifier,assignment,gender,dob,age_in_years,site_id,country,study,vl_date_endline,vl_endline,...,dm_dx_date,dm,multi,bp_controlled_baseline,glucose_controlled_baseline,vl_controlled_baseline,bp_controlled_endline,glucose_controlled_endline,vl_controlled_endline,cohort
0,103-115-0001-0,intervention,F,1955-06-15,65,115,uganda,inte_africa,NaT,NaN,...,NaN,0,0,1.0,NaN,NaN,0.0,NaN,NaN,HTN_ALONE
1,103-111-0001-9,control,F,1968-06-15,52,111,uganda,inte_africa,NaT,NaN,...,NaN,0,0,0.0,NaN,NaN,1.0,NaN,NaN,HTN_ALONE
2,103-115-0002-8,intervention,M,1960-01-01,61,115,uganda,inte_africa,NaT,NaN,...,,1,2,0.0,0.0,NaN,0.0,0.0,NaN,HTN_DM
3,103-111-0002-7,control,M,1972-04-13,48,111,uganda,inte_africa,NaT,NaN,...,NaN,0,0,0.0,NaN,NaN,0.0,NaN,NaN,HTN_ALONE
4,103-115-0003-6,intervention,F,1951-02-06,69,115,uganda,inte_africa,NaT,NaN,...,,1,2,0.0,0.0,NaN,1.0,0.0,NaN,HTN_DM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7024,103-117-0173-3,intervention,M,1963-12-07,57,117,uganda,inte_africa,NaT,NaN,...,NaN,0,0,1.0,NaN,NaN,1.0,NaN,NaN,HTN_ALONE
7025,103-117-0174-1,intervention,F,1961-07-02,60,117,uganda,inte_africa,NaT,NaN,...,,1,2,0.0,1.0,NaN,0.0,0.0,NaN,HTN_DM
7026,103-212-0238-3,intervention,M,1971-08-15,50,212,tanzania,inte_africa,NaT,NaN,...,2021-04-01,1,0,NaN,0.0,NaN,NaN,0.0,NaN,DM_ALONE
7027,103-117-0175-8,intervention,F,1961-01-01,60,117,uganda,inte_africa,NaT,NaN,...,,1,2,1.0,0.0,NaN,0.0,1.0,NaN,HTN_DM
